# 07 — Vietnamese Symptom Extraction: Rule-based, Semantic và Hybrid

## 1. Câu hỏi nghiên cứu

Thí nghiệm so sánh nhận diện triệu chứng tiếng Việt bằng quy tắc, semantic candidate retrieval và hybrid. Đây là bài toán NLP riêng với phân loại bệnh; không huấn luyện LR hay mở final disease test.

**Cách chạy:** chọn kernel Colab rồi chạy các cell từ trên xuống. Hai switch bên dưới đã bật. Setup tự chuẩn bị source theo cấu trúc repository và dependencies NLP. Lần chạy Semantic đầu tiên tải embedding công khai từ publisher theo revision cố định; các lần sau dùng cache đã kiểm checksum trong `artifacts/embeddings/`. Cần kết nối Internet cho lần chuẩn bị đầu. Không cần API key, mount Drive, nhập đường dẫn hoặc upload từng file.

Nếu dùng Python local, mở notebook trong project và cài `requirements-nlp-research.txt` trong môi trường nghiên cứu riêng; notebook không tự cài đè môi trường API LR. Kết quả mỗi lần chạy nằm trong `reports/nlp_research_<timestamp>/`, bảng/hình phía dưới đọc chính thư mục này. Đặt RUN_RESEARCH=False chỉ khi muốn xem snapshot đối chiếu đã lưu.


In [ ]:
RUN_RESEARCH = True
RUN_SEMANTIC = True


In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util, importlib.metadata

# Use the same repository structure on Windows and Colab. No individual uploads.
IS_COLAB = sys.platform.startswith('linux') and Path('/content').is_dir()
REPOSITORY = 'https://github.com/Leafguy2341/disease-prediction-ml.git'
SOURCE_BRANCH = 'feat/vietnamese-nlp-research'
REQUIRED_SOURCE = ('configs/nlp_research.json', 'scripts/nlp_symptom_research.py',
                   'scripts/plot_nlp_research.py', 'requirements-nlp-research.txt',
                   'reports/nlp_research_v1/figure_manifest.json',
                   'data/evaluation/vietnamese_nlp_reserved_v1.jsonl',
                   'artifacts/metadata/phase2/feature_schema.json',
                   'artifacts/metadata/phase2/label_mapping.json')
candidates = [Path.cwd(), *Path.cwd().parents]
if os.environ.get('NLP_PROJECT_ROOT'):
    candidates.insert(0, Path(os.environ['NLP_PROJECT_ROOT']).expanduser())
candidates.extend((Path.cwd()/'disease-prediction-ml-git-sync', Path.cwd()/'disease-prediction-ml'))
SOURCE = next((p.resolve() for p in candidates if all((p/n).is_file() for n in REQUIRED_SOURCE)), None)
if SOURCE is None:
    if not IS_COLAB:
        raise RuntimeError('Mở notebook từ project trong VS Code và chọn kernel Python nghiên cứu. Source cần có configs/, scripts/ và data/evaluation/.')
    SOURCE = Path('/content/disease-prediction-ml-nlp')
    if not SOURCE.exists():
        print('Đang chuẩn bị source GitHub trong Colab...', flush=True)
        subprocess.run(['git','clone','--branch',SOURCE_BRANCH,'--single-branch',REPOSITORY,str(SOURCE)], check=True, capture_output=True, text=True)
    elif (SOURCE/'.git').is_dir():
        remote = subprocess.check_output(['git','-C',str(SOURCE),'remote','get-url','origin'],text=True).strip()
        if remote.removesuffix('.git') != REPOSITORY.removesuffix('.git'):
            raise RuntimeError('Folder source Colab thuộc repository khác; không ghi đè.')
        dirty = subprocess.check_output(['git','-C',str(SOURCE),'status','--porcelain'],text=True).strip()
        if dirty:
            raise RuntimeError('Source Colab có thay đổi local; giữ nguyên để tránh ghi đè. Dùng runtime mới.')
        subprocess.run(['git','-C',str(SOURCE),'fetch','origin',SOURCE_BRANCH],check=True,capture_output=True,text=True)
        subprocess.run(['git','-C',str(SOURCE),'merge','--ff-only','FETCH_HEAD'],check=True,capture_output=True,text=True)
    missing = [n for n in REQUIRED_SOURCE if not (SOURCE/n).is_file()]
    if missing:
        raise RuntimeError('Source chưa đầy đủ: '+', '.join(missing))
if SOURCE is not None:
    os.environ['NLP_PROJECT_ROOT'] = str(SOURCE)
    print('Source:', SOURCE)
    if (SOURCE/'.git').is_dir():
        print('Commit:', subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip())
if RUN_RESEARCH and RUN_SEMANTIC:
    expected = {'sentence-transformers':'5.1.2', 'transformers':'4.57.1', 'huggingface-hub':'0.36.0'}
    def package_version(name):
        try: return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: return None
    if any(package_version(name)!=version for name,version in expected.items()):
        if not IS_COLAB:
            raise RuntimeError('Cài requirements-nlp-research.txt bằng interpreter nghiên cứu riêng; không dùng môi trường API LR. Colab tự chuẩn bị dependencies ở cell này.')
        print('Đang chuẩn bị thư viện NLP cho runtime Colab...', flush=True)
        install = subprocess.run([sys.executable,'-m','pip','install','--quiet','-r',str(SOURCE/'requirements-nlp-research.txt')],capture_output=True,text=True)
        if install.returncode:
            raise RuntimeError('Cài dependencies NLP thất bại. '+install.stderr[-1600:])
    print('Thư viện NLP:', {name:package_version(name) for name in expected})

import json, hashlib
from IPython.display import display, Markdown, Image

ROOT = SOURCE
REPORT = ROOT / 'reports/nlp_research_v1'
reference_manifest = json.loads((REPORT / 'figure_manifest.json').read_text(encoding='utf-8'))

def snapshot_bytes(name):
    if name not in reference_manifest['files']:
        raise ValueError('Unknown saved report file: ' + name)
    raw = (REPORT / name).read_bytes()
    if hashlib.sha256(raw).hexdigest() != reference_manifest['files'][name]:
        raise ValueError('Saved report checksum mismatch: ' + name)
    return raw

def snapshot_json(name):
    return json.loads(snapshot_bytes(name).decode('utf-8'))

protocol = json.loads((ROOT / 'configs/nlp_research.json').read_text(encoding='utf-8'))
saved = snapshot_json('benchmark.json')
inventory = snapshot_json('coverage_inventory.json')
after = snapshot_json('context_after_fix.json')
print('Source và dữ liệu đối chiếu đã sẵn sàng. Kết quả lần chạy mới hiện ở mục 6.')


## 2. Benchmark và provenance

86 fixtures tiếng Việt lịch sử chỉ dùng development; 63 câu có expected states đầy đủ, 23 câu chỉ gán nhãn từng phần/forbidden features. Không coi phần chưa gán nhãn là negative. Thêm 32 câu reserved synthetic v1, pending_human_review, không dùng trong alias probes/threshold selection; exact normalized text không giao nhau giữa hai tập. Cùng tác giả và khái niệm triệu chứng vẫn tạo bias: đây không phải bộ bệnh nhân độc lập hoặc clinical validation.

Mỗi ca mới ghi canonical states, subject=self, ambiguity, evidence offsets khi có, provenance và flags tách evaluation khỏi development. Các ca other-subject/hư cấu kỳ vọng không có state cho người dùng. Fixture lịch sử không có span gold được ghi nhận giới hạn, không bịa span annotation.

**Reserved đã được quan sát trong lần baseline đầu**, phát hiện hai false positive hư cấu. Giữ nguyên kết quả trước sửa; không chạy lại để gọi đó là cải thiện độc lập. Cần bộ NLP evaluation mới được người duyệt trước kết luận tổng quát hoặc chọn threshold. Đây không phải locked final disease test.


In [ ]:
display(protocol)
for split, methods in saved['results'].items():
    m = methods['rule_based']['metrics']
    print(split, '| cases:', m['cases'], '| fully annotated:', m['fully_annotated_cases'], '| partial:', m['partial_label_cases'])
display(saved['inputs_sha256'])
display(saved['baseline_source_sha256'])


## 3. Existing Rule-based baseline và inventory 375 feature

Helper gọi production `OfflineConversation.handle` → `OfflineExtractor` → canonical validator thật; không viết bản rule đơn giản để thay baseline, không cung cấp prediction service. Chỉ đọc trạng thái/evidence, kiểm span slices và giải phóng session từng ca.

Inventory giữ schema order, display_vi, mô tả phân biệt, aliases, nguồn/chưa xác minh, collision, forbidden contexts và trạng thái production. DRAFT = dự thảo; LEXICALLY_REVIEWED = rà từ vựng nội bộ, không thẩm định y khoa; CONTEXT_TESTED = chỉ các template đã thử; PRODUCTION_ACTIVE = đang được runtime dùng; REVIEW_REQUIRED = còn blocker. Các mức này không tương đương clinical review.

375/375 đã có tên và ghi chú trước nhiệm vụ; không gọi đó là 375/375 production. Probes mỗi feature dùng dictionary tạm, không sửa synonyms. Kết quả sau sửa guard lưu riêng; các feature còn semantic overlap/decision HOLD/CLARIFY/REVISE không được kích hoạt. Việc qua vài template cũng chưa thay cho batch release regression.


In [ ]:
display(inventory['summary'])
display(after['summary'])
display([dict(feature=r['canonical_feature'], display_vi=r['display_vi'], lifecycle=r['lifecycle'], blockers=r['activation_blockers'])
         for r in after['features'][:8]])
print('Catalog đầy đủ:', len(after['features']), 'features trong snapshot; CSV source:', REPORT / 'canonical_375.csv' if REPORT else 'reports/nlp_research_v1/canonical_375.csv trong repository')


## 4. Semantic matching

Embedding: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, revision `e8f8c211226b894fcb81acc59f3b34ba3efd5f42`, Apache-2.0. [Model card](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2). Publisher liệt kê tiếng Việt; mô hình dùng cho phrase/sentence similarity, không phải classifier bệnh.

Cell chạy bên dưới tự tải các file public của đúng revision vào `artifacts/embeddings/`, tạo inventory SHA-256 và kiểm lại trước khi load. Chỉ dùng safetensors, `trust_remote_code=False`, không API key. Snapshot sai checksum dừng ngay, không ghi đè. Không sửa model LR, metadata Phase2 hay từ điển production.

Truy hồi Top3 theo từng clause có evidence offsets và kiểm subject/assertion/time. Cosine chỉ dùng xếp hạng ứng viên. Ứng viên semantic chờ xác nhận, không tự biến thành triệu chứng trong vector bệnh. Vì vậy báo riêng retrieval Recall@1/Recall@3/Precision@3 và precision/recall/F1 của trạng thái đã xác nhận. Không chọn threshold trên tập evaluation.


In [ ]:
RESULT_FOLDER = None
research = None
if RUN_RESEARCH:
    from datetime import datetime, timezone
    RESULT_FOLDER = ROOT / 'reports' / ('nlp_research_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    command = [sys.executable, '-B', '-m', 'scripts.nlp_symptom_research', '--output', str(RESULT_FOLDER)]
    if RUN_SEMANTIC:
        command.append('--auto-embedding')
    print('Đang chạy extraction và tính metrics. Chuẩn bị embedding lần đầu có thể mất vài phút.', flush=True)
    completed = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
    if completed.returncode:
        raise RuntimeError('Thí nghiệm dừng trước khi có kết quả hoàn tất:\n' + completed.stderr[-2200:])
    research = json.loads((RESULT_FOLDER / 'benchmark.json').read_text(encoding='utf-8'))
    new_inventory = json.loads((RESULT_FOLDER / 'coverage_inventory.json').read_text(encoding='utf-8'))
    print('Kết quả lần chạy:', RESULT_FOLDER)
    print('Embedding:', research['semantic_environment']['status'])
else:
    print('Đang xem snapshot đối chiếu; không chạy extraction mới.')


## 5. Hybrid matching

Hybrid giữ nguyên validated rule states; retrieval chỉ bổ sung ứng viên chưa có trong rule state và yêu cầu confirmation. Semantic không override denied/uncertain hoặc other-subject/fiction guard. Nghiên cứu dùng lại `evidence_context` để lọc evidence theo subject/time/assertion; điều đó chỉ chứng minh context, không chứng minh alias đồng nghĩa.

Không giả lập câu trả lời “có” để tăng recall. Khi chưa có người xác nhận, semantic-only abstain; hybrid chỉ có committed states từ rules. Benchmark tương lai phải phân biệt candidate recall@K với recall của states đã xác nhận, và đánh giá thêm user confirmation thật. Không có tích hợp semantic production trong nhiệm vụ này.


## 6. Evaluation và mẫu số

Precision/Recall/Micro F1 tính trên cặp `(canonical feature, assertion status)` của câu fully annotated; zero_division=0. Macro F1 trung bình trên union cặp gold/pred đã quan sát, báo denominator; không phải macro 587 disease labels hay coverage đủ 375 triệu chứng. Exact-set match so toàn bộ states của câu fully annotated. Partial cases chỉ kiểm forbidden-feature violations, không đưa vào P/R/F1.

False positive breakdown ghi theo loại context; FN ghi riêng. Mapping coverage là số feature khác nhau được nhận diện / 375, không phải activation coverage. Confirmation rate là tỷ lệ câu cần clarification, không phải xác nhận đúng hoặc thành công hội thoại. Latency p50/p95 đo một lượt trên máy này, không warm-up/multi-user benchmark hoặc cam kết SLA.


In [ ]:
import pandas as pd
evaluated = research if research is not None else saved
print('Nguồn kết quả:', RESULT_FOLDER if RESULT_FOLDER is not None else 'Snapshot lịch sử có checksum')
rows, retrieval = [], []
for split, methods in evaluated['results'].items():
    for method, result in methods.items():
        rows.append({'split': split, 'method': method, 'status': result['status'], **result.get('metrics', {})})
        if 'candidate_metrics' in result:
            retrieval.append({'split': split, 'method': method, **result['candidate_metrics']})
display(pd.DataFrame(rows).drop(columns=['false_positive_contexts', 'macro_definition', 'confirmation_definition'], errors='ignore'))
if retrieval:
    print('Đánh giá truy hồi ứng viên — không phải trạng thái đã xác nhận:')
    display(pd.DataFrame(retrieval))
display({split: methods['rule_based']['metrics']['false_positive_contexts'] for split, methods in evaluated['results'].items()})


## 7. Biểu đồ thí nghiệm

Khi chạy mới, các hình được tạo trực tiếp từ `benchmark.json` của cùng run: extraction states, candidate retrieval và latency. Mỗi file hình có SHA trong `run_figures.json`. Semantic candidates chưa xác nhận không được cộng vào vector hay báo như confirmed extraction. Ở chế độ xem snapshot, hình lịch sử được giữ nguyên nhãn và checksum.


In [ ]:
if RESULT_FOLDER is not None:
    manifest = json.loads((RESULT_FOLDER / 'run_figures.json').read_text(encoding='utf-8'))
    if hashlib.sha256((RESULT_FOLDER / 'benchmark.json').read_bytes()).hexdigest() != manifest['benchmark_sha256']:
        raise ValueError('Metrics/figure manifest mismatch')
    for name, expected in manifest['figures'].items():
        raw = (RESULT_FOLDER / name).read_bytes()
        if hashlib.sha256(raw).hexdigest() != expected:
            raise ValueError('Figure checksum mismatch: ' + name)
        display(Image(data=raw, format='png'))
else:
    for name in ('rule_metrics.png', 'context_errors.png', 'activation_coverage.png', 'rule_latency.png'):
        display(Image(data=snapshot_bytes(name), format='png'))


## 8. Kết luận và giới hạn

Đánh giá các phương án theo kết quả đo trong cùng run: mức cải thiện recall cần được xem cùng precision, lỗi phủ định/chủ thể, tỷ lệ cần xác nhận và độ trễ. Semantic retrieval trả về ứng viên theo độ tương đồng; Hybrid kết hợp các ứng viên này với trạng thái đã kiểm chứng từ quy tắc. Không suy chất lượng phân loại bệnh từ cosine similarity.

Baseline lưu ban đầu cho thấy lỗi hư cấu trên reserved synthetic set; guard được sửa theo câu chứa evidence và có regression riêng. Giữ kết quả lịch sử để đối chiếu phiên bản, không coi rerun trên tập đã quan sát là bằng chứng tổng quát hóa độc lập.

Giữ nguyên 135 production features cho đến khi aliases vượt context, semantic-overlap và batch release checks; inventory đầy đủ không tự cho phép activation. Các giả định của nhãn tổng hợp, collision không dấu, typo, phủ định phức tạp, đồng nghĩa và cách nói ngoài catalog vẫn cần người duyệt. 32 câu synthetic không đủ chứng minh chất lượng extraction ngoài đời.

Xem [README](../README.md), [NLP contract](../docs/phase5_llm_integration_design.md), [development log](../docs/DEVELOPMENT_LOG.md). Notebook 01–06, model LR tuned, VT318, schema375/class587 và scientific results không được sửa bởi nghiên cứu này.
